In [85]:
import os
import pandas as pd

In [86]:
emission = pd.read_csv (r"C:\Users\mhtaba\Desktop\pod_lca_git\pod_lca\data\transportation_dataset\Emission.csv")
data_material = pd.read_csv (r"C:\Users\mhtaba\Desktop\pod_lca_git\pod_lca\data\transportation_dataset\EC3 Category to CFS Group mapping.csv")
cfs = pd.read_csv (r"C:\Users\mhtaba\Desktop\pod_lca_git\pod_lca\data\transportation_dataset\cfs_2017.csv")

In [87]:
df

,SHIPMT_ID,ORIG_STATE,ORIG_MA,ORIG_CFS_AREA,DEST_STATE,DEST_MA,DEST_CFS_AREA,NAICS,QUARTER,SCTG,...,SHIPMT_VALUE,SHIPMT_WGHT,SHIPMT_DIST_GC,SHIPMT_DIST_ROUTED,TEMP_CNTL_YN,EXPORT_YN,EXPORT_CNTRY,HAZMAT,WGT_FACTOR,quartile
19,20,17,176,17-176,27,378,27-378,332,2,32,...,58828,25033,348,406,N,N,N,N,99.8,Q3
23,24,1,99999,01-99999,47,99999,47-99999,4235,3,32,...,19000,42222,102,136,N,N,N,N,341.7,Q2
44,45,20,312,20-312,29,312,29-312,332,2,32,...,736,1307,15,16,N,N,N,N,1102.1,Q1
55,56,48,288,48-288,48,206,48-206,332,1,32,...,13122,6424,197,220,N,N,N,N,212.9,Q3
105,106,12,422,12-422,12,370,12-370,4235,2,32,...,22980,12132,186,220,N,N,N,N,443.5,Q3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048410,1048411,12,422,12-422,51,40060,51-40060,4239,2,32,...,32719,38689,635,710,N,N,N,N,18.6,Q4
1048467,1048468,36,408,36-408,36,408,36-408,331,4,32,...,9839,21866,2,3,N,N,N,N,20.8,Q1
1048472,1048473,19,99999,19-99999,17,99999,17-99999,4235,2,32,...,2023,2326,91,116,N,N,N,N,131.4,Q2
1048491,1048492,18,258,18-258,48,206,48-206,331,3,32,...,12264,45849,882,1190,N,N,N,N,508.3,Q4


In [88]:
data_material

,material,SCTG
0,FireSprinklers,409
1,SmokeDetectors,409
2,AHUs,344
3,RTUs,344
4,AirTerminals,344
...,...,...
234,Profiles,409
235,CleaningProducts,233
236,Clothing,303
237,FoodBeverage,7


In [89]:
sctg = data_material[data_material["material"] == "StructuralSteel"].iloc[0,1]
sctg = int(str(sctg)[:2])
sctg

32

In [90]:
df = cfs[cfs ["SCTG"] == "32"]
df

,SHIPMT_ID,ORIG_STATE,ORIG_MA,ORIG_CFS_AREA,DEST_STATE,DEST_MA,DEST_CFS_AREA,NAICS,QUARTER,SCTG,MODE,SHIPMT_VALUE,SHIPMT_WGHT,SHIPMT_DIST_GC,SHIPMT_DIST_ROUTED,TEMP_CNTL_YN,EXPORT_YN,EXPORT_CNTRY,HAZMAT,WGT_FACTOR
19,20,17,176,17-176,27,378,27-378,332,2,32,4,58828,25033,348,406,N,N,N,N,99.8
23,24,1,99999,01-99999,47,99999,47-99999,4235,3,32,4,19000,42222,102,136,N,N,N,N,341.7
44,45,20,312,20-312,29,312,29-312,332,2,32,5,736,1307,15,16,N,N,N,N,1102.1
55,56,48,288,48-288,48,206,48-206,332,1,32,4,13122,6424,197,220,N,N,N,N,212.9
105,106,12,422,12-422,12,370,12-370,4235,2,32,5,22980,12132,186,220,N,N,N,N,443.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048410,1048411,12,422,12-422,51,40060,51-40060,4239,2,32,4,32719,38689,635,710,N,N,N,N,18.6
1048467,1048468,36,408,36-408,36,408,36-408,331,4,32,5,9839,21866,2,3,N,N,N,N,20.8
1048472,1048473,19,99999,19-99999,17,99999,17-99999,4235,2,32,4,2023,2326,91,116,N,N,N,N,131.4
1048491,1048492,18,258,18-258,48,206,48-206,331,3,32,15,12264,45849,882,1190,N,N,N,N,508.3


In [91]:
quartiles = df["SHIPMT_DIST_ROUTED"].quantile([0.25, 0.5, 0.75]).values

In [92]:
quartiles

array([ 27., 142., 498.])

In [93]:
def assign_quartile(x, q1, q2, q3):

    if x <= q1:
        return 'Q1'
    elif x <= q2:
        return 'Q2'
    elif x <= q3:
        return 'Q3'
    else:
        return 'Q4'

df['quartile'] = df["SHIPMT_DIST_ROUTED"].apply(assign_quartile, args=(quartiles[0], quartiles[1], quartiles[2]))

C:\Users\mhtaba\AppData\Local\Temp\ipykernel_14888\1949202098.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['quartile'] = df["SHIPMT_DIST_ROUTED"].apply(assign_quartile, args=(quartiles[0], quartiles[1], quartiles[2]))


In [94]:
local = df[df['quartile'] == 'Q1']
regional = df[df['quartile'] == 'Q2']
regional_c = df[df['quartile'] == 'Q3']
national = df[df['quartile'] == 'Q4']

In [95]:
local

,SHIPMT_ID,ORIG_STATE,ORIG_MA,ORIG_CFS_AREA,DEST_STATE,DEST_MA,DEST_CFS_AREA,NAICS,QUARTER,SCTG,...,SHIPMT_VALUE,SHIPMT_WGHT,SHIPMT_DIST_GC,SHIPMT_DIST_ROUTED,TEMP_CNTL_YN,EXPORT_YN,EXPORT_CNTRY,HAZMAT,WGT_FACTOR,quartile
44,45,20,312,20-312,29,312,29-312,332,2,32,...,736,1307,15,16,N,N,N,N,1102.1,Q1
106,107,6,99999,06-99999,6,99999,06-99999,4235,4,32,...,91,41,3,3,N,N,N,N,961.2,Q1
107,108,18,99999,18-99999,18,99999,18-99999,331,3,32,...,2771,1056,10,10,N,N,N,N,818.4,Q1
241,242,42,99999,42-99999,24,47900,24-47900,332,4,32,...,1674,440,22,25,N,N,N,N,661.4,Q1
267,268,41,440,41-440,41,440,41-440,4235,4,32,...,613,1578,19,23,N,N,N,N,279.9,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1047898,1047899,28,99999,28-99999,28,99999,28-99999,4239,2,32,...,6498,39343,5,6,N,N,N,N,783.0,Q1
1047935,1047936,42,99999,42-99999,42,99999,42-99999,4233,4,32,...,419,23,4,6,N,N,N,N,3737.2,Q1
1048089,1048090,39,212,39-212,39,212,39-212,4235,2,32,...,248,57,5,5,N,N,N,N,596.8,Q1
1048283,1048284,6,99999,06-99999,6,99999,06-99999,4233,3,32,...,351,863,8,8,N,N,N,N,276.1,Q1


In [96]:
local_dis = local["SHIPMT_DIST_ROUTED"].mean()

In [97]:
local = pd.merge (emission, local, on ="MODE")


In [99]:
local

,mode_name,MODE,eff,Units,GWP,AP,EP,ODP,SFP,SHIPMT_ID,...,SHIPMT_VALUE,SHIPMT_WGHT,SHIPMT_DIST_GC,SHIPMT_DIST_ROUTED,TEMP_CNTL_YN,EXPORT_YN,EXPORT_CNTRY,HAZMAT,WGT_FACTOR,quartile
0,Truck,4,1,tkm,9,3,8,9,8,107,...,91,41,3,3,N,N,N,N,961.2,Q1
1,Truck,4,1,tkm,9,3,8,9,8,684,...,1285,525,5,5,N,N,N,N,9.5,Q1
2,Truck,4,1,tkm,9,3,8,9,8,882,...,3602,32422,22,26,N,N,N,N,130.6,Q1
3,Truck,4,1,tkm,9,3,8,9,8,988,...,7,11,1,1,N,N,N,N,1967.4,Q1
4,Truck,4,1,tkm,9,3,8,9,8,1496,...,2227,95,2,4,N,N,N,N,162.5,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15899,Air,5,1,tkm,4,10,6,5,7,1047435,...,23,41,3,3,N,N,N,N,125.6,Q1
15900,Air,5,1,tkm,4,10,6,5,7,1047475,...,60,18,9,10,N,N,N,N,3883.0,Q1
15901,Air,5,1,tkm,4,10,6,5,7,1047864,...,1845,1132,18,24,N,N,N,N,246.6,Q1
15902,Air,5,1,tkm,4,10,6,5,7,1047936,...,419,23,4,6,N,N,N,N,3737.2,Q1


In [100]:
merged_df['GWP'] = merged_df['GWP'] * merged_df['distance']
merged_df['AP'] = merged_df['AP'] * merged_df['distance']
merged_df['EP'] = merged_df['EP'] * merged_df['distance']
merged_df['ODP'] = merged_df['ODP'] * merged_df['distance']
merged_df['SFP'] = merged_df['SFP'] * merged_df['distance']

In [101]:
local_impact = merged_df[['mode', 'GWP', 'AP', 'EP', 'ODP', 'SFP']]

In [102]:
final = local_impact.iloc[0].to_dict()

In [103]:
final

{'mode': 'Truck',
 'GWP': 9000000,
 'AP': 3000000,
 'EP': 8000000,
 'ODP': 9000000,
 'SFP': 8000000}